# Figures of the paper
**Machine Learning Based Optimal Control of Delayed Sobolev-Type Fractional Evolution Systems of Order (1,2) under Seismic Disturbances**

M. Siddiq and A. U. K. Niazi, Department of Mathematics and Statistics, The University of Lahore, Sargodha

This notebook draws all seven figures of Section 6. Each figure has its own cell and can be run separately (after Steps A and B). Every cell takes only a few seconds.

* **Figures 1 and 4** are computed here directly (Figure 4 runs the ABM solver for 17 fractional orders).
* **Figures 2, 3, 5, 6 and 7** are drawn from the saved numerical results of the full computation (fPINN training, NN open-loop solver training and the direct method). These trainings take about one hour; the full computation is in the separate notebook `Seismic_Fractional_Control_Colab.ipynb`.

## Step A. Libraries and plotting style

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import FancyArrowPatch, Rectangle
torch.set_default_dtype(torch.float64)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False, "figure.dpi": 110})
C = dict(blue="#005AB4", red="#B3261E", green="#2E7D32", amber="#B8860B", grey="#555555")
import os; os.makedirs("figures", exist_ok=True)
print("Libraries ready")

## Step B. Classical ABM solver (spectral Galerkin in space, Adams–Bashforth–Moulton predictor–corrector in time)

In [ ]:
import math
# ---------------- parameters ----------------
P = dict(rho=1.6, a=0.1, c=1.0, zeta=0.1, c_g=0.1, t1=2.0, tau=0.2, kappa=0.1,
         Theta=4.0, h=0.02, N=8, Nw=64, A0=0.05, rho_u=1e-4, umax=10.0, eqscale=4.0)

def setup(p):
    N, Nw = p["N"], p["Nw"]
    n = np.arange(1, N + 1)
    lam = (n * np.pi) ** 2
    Ediag = 1 + p["a"] * lam                     # eigenvalues of E
    om2 = p["c"] ** 2 * lam / Ediag              # -eigenvalues of G E^{-1}
    w = (np.arange(Nw) + 0.5) / Nw               # midpoint grid
    Phi = np.sqrt(2) * np.sin(np.outer(w, n * np.pi))   # (Nw,N) basis values
    Wq = Phi.T / Nw                               # projection: coeffs = Wq @ values
    b = np.exp(-((w - 0.5) / 0.1) ** 2)           # actuator profile
    m = np.ones(Nw)                               # uniform inertial load
    return dict(n=n, lam=lam, E=Ediag, om2=om2, w=w, Phi=Phi, Wq=Wq,
                b_n=Wq @ b, m_n=Wq @ m, b=b, m=m)

def weights(M, h, al):
    """DFF predictor B and corrector A weights for order al (1<al<2)."""
    k = np.arange(M + 1)
    B = np.zeros((M + 1, M + 1)); A = np.zeros((M + 1, M + 1)); Adiag = np.zeros(M + 1)
    c = h ** al / (al * (al + 1))
    for kk in range(M):  # step kk -> kk+1
        j = np.arange(kk + 1)
        B[kk + 1, :kk + 1] = h ** al / al * ((kk + 1 - j) ** al - (kk - j) ** al)
        a = np.zeros(kk + 1)
        a[0] = kk ** (al + 1) - (kk - al) * (kk + 1) ** al
        jj = np.arange(1, kk + 1)
        a[1:] = (kk - jj + 2) ** (al + 1) + (kk - jj) ** (al + 1) - 2 * (kk - jj + 1) ** (al + 1)
        A[kk + 1, :kk + 1] = c * a
        Adiag[kk + 1] = c
    return B / math.gamma(al), A / math.gamma(al), Adiag / math.gamma(al)

class Solver:
    def __init__(self, p):
        self.p = p; self.S = setup(p)
        self.M = int(round(p["Theta"] / p["h"])); self.t = np.arange(self.M + 1) * p["h"]
        B, A, Ad = weights(self.M, p["h"], p["rho"])
        T = lambda x: torch.tensor(x)
        self.B, self.A, self.Ad = T(B), T(A), T(Ad)
        S = self.S
        self.E, self.om2 = T(S["E"]), T(S["om2"])
        self.Phi, self.Wq = T(S["Phi"]), T(S["Wq"])
        self.b_n, self.m_n = T(S["b_n"]), T(S["m_n"])
        self.dtau = int(round(p["tau"] / p["h"])); self.dkap = int(round(p["kappa"] / p["h"]))
        self.i1 = int(round(p["t1"] / p["h"]))
        phi = p["A0"] * np.sin(np.pi * S["w"])        # constant history shape
        self.phi_n = T(S["Wq"] @ phi)

    def rhs(self, y, ydel, u, v):
        # modal right-hand side of D^rho y = -om2 y + F_n / E_n
        xdel = self.Phi @ ydel
        J = -self.p["zeta"] * torch.tanh(xdel)
        F = self.Wq @ J + self.b_n * u - self.m_n * v
        return -self.om2 * y + F / self.E

    def solve_given_g(self, g_n, u, v):
        """u,v: tensors of length M+1 (u already delayed-consistent: u[k]=mu(t_k-kappa))."""
        y0 = self.phi_n + g_n
        Y = [y0]; Fs = [self.rhs(y0, y0, u[0], v[0])]
        for k in range(self.M):
            Fst = torch.stack(Fs)
            kd = k + 1 - self.dtau
            ydel = Y[kd] if kd >= 0 else y0
            pred = y0 + self.B[k + 1, :k + 1] @ Fst
            fpred = self.rhs(pred, ydel, u[k + 1], v[k + 1])
            corr = y0 + self.A[k + 1, :k + 1] @ Fst + self.Ad[k + 1] * fpred
            Y.append(corr); Fs.append(self.rhs(corr, ydel, u[k + 1], v[k + 1]))
        return torch.stack(Y)          # (M+1, N)

    def solve(self, mu, varpi, psi=None, iters=30, tol=1e-12):
        """mu: control on grid t_k (length M+1); delayed by kappa using history psi (default 0)."""
        dk = self.dkap
        hist = torch.zeros(dk) if psi is None else psi
        u = torch.cat([hist, mu[:self.M + 1 - dk]]) if dk > 0 else mu
        g = torch.zeros_like(self.phi_n)
        for _ in range(iters):
            Y = self.solve_given_g(g, u, varpi)
            g_new = self.p["c_g"] * Y[self.i1] / self.E      # g = c_g E^{-1} xi(t1)
            if torch.max(torch.abs(g_new - g)) < tol: g = g_new; break
            g = g_new.detach() if not g_new.requires_grad else g_new
        Y = self.solve_given_g(g, u, varpi)
        return Y, g

    def l2norm_t(self, Y):  # ||xi(t)||_{L2} = Euclidean norm of modal coefficients
        return torch.sqrt(torch.sum(Y ** 2, dim=1))

    def supnorm(self, Y, g):
        hist = torch.sqrt(torch.sum((self.phi_n + g) ** 2))
        return torch.maximum(self.l2norm_t(Y).max(), hist)

    def cost(self, Y, mu):
        h = self.p["h"]
        return h * torch.sum(self.l2norm_t(Y) ** 2 + self.p["rho_u"] * mu ** 2)

print('ABM solver ready')

## Step C. Saved numerical results of the full computation

In [ ]:
import base64, io, numpy as np
DATA_B64 = (
"..."
)
DATA = np.load(io.BytesIO(base64.b64decode(DATA_B64)))
print('Saved numerical results loaded:', len(DATA.files), 'arrays')

## Figure 1. The model problem

In [ ]:
a, c, A0 = 0.1, 1.0, 0.05
w = np.linspace(0, 1, 400)
b = np.exp(-((w - 0.5) / 0.1) ** 2)
phi = A0 * np.sin(np.pi * w)
n = np.arange(1, 9); lam = (n * np.pi) ** 2; om = np.sqrt(c ** 2 * lam / (1 + a * lam))
BLUE, AMBER, GREY, RED = "#2a788e", "#e8a202", "#555555", "#b3261e"

fig = plt.figure(figsize=(7.2, 4.9))
gs = fig.add_gridspec(2, 2, height_ratios=[0.95, 1], hspace=0.38, wspace=0.32)

# (a) schematic
ax = fig.add_subplot(gs[0, :]); ax.set_xlim(-0.12, 1.12); ax.set_ylim(-0.56, 0.62); ax.axis("off")
ax.fill_between([-0.12, 1.12], -0.55, -0.42, color="#d9d9d9", lw=0)
ax.plot([-0.12, 1.12], [-0.42, -0.42], color=GREY, lw=1)
for x0 in [-0.04, 1.0]:
    ax.add_patch(Rectangle((x0, -0.42), 0.04, 0.42, color=GREY))
defl = 0.22 * np.sin(np.pi * w)
ax.plot(w, 0 * w, color="#bbbbbb", lw=1.2, ls="--")
ax.plot(w, defl, color=BLUE, lw=3)
ax.annotate(r"$\xi(t,w)$", (0.25, 0.22 * np.sin(np.pi * 0.25)), xytext=(0.12, 0.42),
            arrowprops=dict(arrowstyle="-", color=GREY, lw=0.8), fontsize=10, color=BLUE)
ax.add_patch(FancyArrowPatch((0.5, 0.58), (0.5, 0.26), arrowstyle="-|>", mutation_scale=14, color=AMBER, lw=2))
ax.text(0.53, 0.45, r"actuator $b(w)\,\mu(t-\kappa)$", color="#8a5d00", fontsize=10)
ax.add_patch(FancyArrowPatch((0.22, -0.49), (0.40, -0.49), arrowstyle="<|-|>", mutation_scale=12, color=RED, lw=1.6))
ax.text(0.43, -0.52, r"ground acceleration $\varpi(t)$", color=RED, fontsize=10)
ax.text(-0.11, 0.08, r"$\xi(t,0)=0$", fontsize=9, color=GREY)
ax.text(0.95, 0.08, r"$\xi(t,1)=0$", fontsize=9, color=GREY)
ax.text(0.72, 0.28, r"$-\zeta\tanh\,\xi(t-\tau,w)$", fontsize=9, color=GREY)
ax.set_title(r"(a) viscoelastic structure on $w\in(0,1)$ with fixed ends", fontsize=9)

# (b) spatial profiles
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(w, b, color=AMBER, lw=2, label=r"actuator profile $b(w)$")
ax2.plot(w, phi / A0, color=BLUE, lw=2, ls="--", label=r"history shape $\phi(w)/A_0$")
ax2.set_xlabel(r"$w$"); ax2.set_ylabel("profile"); ax2.set_ylim(0, 1.6); ax2.grid(alpha=0.3)
ax2.legend(fontsize=8, loc="upper center", ncol=1); ax2.set_title("(b) spatial profiles", fontsize=9)

# (c) natural frequencies
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(n, om, "o-", color=BLUE, lw=1.8, ms=6, mec="white", mew=1.0, label=r"$\omega_n=\sqrt{c^2\lambda_n/(1+a\lambda_n)}$")
ax3.axhline(c / np.sqrt(a), color=GREY, lw=1, ls=":", label=r"limit $c/\sqrt{a}=3.16$")
ax3.annotate(f"$\\omega_1={om[0]:.2f}$", (1, om[0]), xytext=(1.6, 1.35), fontsize=8,
             arrowprops=dict(arrowstyle="-", color=GREY, lw=0.8))
ax3.set_xlabel(r"mode $n$"); ax3.set_ylabel(r"$\omega_n$"); ax3.set_ylim(0, 3.6); ax3.grid(alpha=0.3)
ax3.legend(fontsize=8, loc="lower right"); ax3.set_title("(c) natural frequencies", fontsize=9)

fig.savefig("figures/fig1_model.png", bbox_inches="tight", dpi=200); plt.show()
print("omega_1 =", round(om[0], 3))

## Figure 2. ABM solution and fPINN error (smooth forcing ϖ = 0.5 sin 2t)

In [ ]:
t, w = DATA["s_t"], DATA["s_w"]; xref, xpinn = DATA["s_xref"], DATA["s_xpinn"]
err = np.abs(xref - xpinn)
print("fPINN relative L2 error:", round(100 * np.linalg.norm(xref - xpinn) / np.linalg.norm(xref), 2), "%")
fig = plt.figure(figsize=(7.2, 2.9))
ax = fig.add_subplot(1, 2, 1, projection="3d")
WW, TT = np.meshgrid(w, t)
ax.plot_surface(TT, WW, xref, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel(r"$t$"); ax.set_ylabel(r"$w$"); ax.set_zlabel(r"$\xi(t,w)$"); ax.set_title("(a) ABM reference solution", fontsize=9)
ax.view_init(28, -125)
ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contourf(TT, WW, err, 30, cmap="magma"); fig.colorbar(cs, ax=ax2, fraction=0.046)
ax2.set_xlabel(r"$t$"); ax2.set_ylabel(r"$w$"); ax2.set_title(r"(b) $|\xi_{\mathrm{ABM}}-\xi_{\mathrm{fPINN}}|$", fontsize=9); ax2.grid(False)
fig.tight_layout(); fig.savefig("figures/fig2_solution_error.png", dpi=200); plt.show()

## Figure 3. fPINN midpoint displacement and training loss (smooth forcing)

In [ ]:
t, w = DATA["s_t"], DATA["s_w"]; xref, xpinn = DATA["s_xref"], DATA["s_xpinn"]
iw = int(np.argmin(np.abs(w - 0.5)))
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.7))
axs[0].plot(t, xref[:, iw], color=C["blue"], lw=1.6, label="ABM (classical)")
axs[0].plot(t[::6], xpinn[::6, iw], "o", ms=3.2, mfc="none", color=C["red"], label="fPINN")
axs[0].set_xlabel(r"$t$"); axs[0].set_ylabel(r"$\xi(t,1/2)$"); axs[0].legend(); axs[0].set_title("(a) midpoint displacement", fontsize=9)
hl = DATA["s_hist"]
axs[1].semilogy(hl[:, 0], hl[:, 1], color=C["green"], lw=1.4)
axs[1].set_xlabel("iteration"); axs[1].set_ylabel("loss"); axs[1].set_title("(b) fPINN training loss", fontsize=9)
fig.tight_layout(); fig.savefig("figures/fig3_fpinn.png", dpi=200); plt.show()

## Figure 4. Effect of the fractional order ϱ (computed here with the ABM solver)

In [ ]:
rhos_fine = np.round(np.arange(1.10, 1.901, 0.05), 2)
rhos_show = [1.2, 1.4, 1.6, 1.8]
curves, peaks, energy = {}, [], []
for r in rhos_fine:
    Pr = dict(P); Pr["rho"] = float(r); s = Solver(Pr)
    with torch.no_grad():
        Y, _ = s.solve(torch.zeros(s.M + 1), torch.tensor(0.5 * np.sin(2.0 * s.t)))
    x = (Y @ s.Phi.T).numpy()
    iw = np.argmin(np.abs(s.S["w"] - 0.5))
    curves[float(r)] = x[:, iw]
    peaks.append(np.abs(x[:, iw]).max())
    nrm = np.sqrt(np.sum(Y.numpy() ** 2, axis=1))
    energy.append(np.sqrt(s.p["h"] * np.sum(nrm ** 2)))
t = s.t
cols = ["#440154", "#2a788e", "#22a884", "#e8a202"]    # viridis-like, dark purple -> amber = low -> high order
styles = ["-", "--", "-.", (0, (1, 1.2))]
fig = plt.figure(figsize=(7.2, 5.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1.05, 1], hspace=0.45, wspace=0.32)

ax = fig.add_subplot(gs[0, :])
for r, col, ls in zip(rhos_show, cols, styles):
    ax.plot(t, curves[r], color=col, lw=2.0, ls=ls, label=rf"$\varrho={r}$")
    k = np.argmax(np.abs(curves[r]))
    ax.plot(t[k], curves[r][k], "o", ms=6, color=col, mec="white", mew=1.0, zorder=3)
ax.axhline(0, color="#999999", lw=0.8)
ax.set_xlim(0, 4); ax.set_xlabel(r"$t$"); ax.set_ylabel(r"$\xi(t,1/2)$")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.16), fontsize=9, handlelength=3.0)
ax.set_title("(a) midpoint displacement for four orders (dots: peak values)", fontsize=9, pad=24)

ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(rhos_fine, peaks, "-", color="#2a788e", lw=2.0)
ax2.plot(rhos_fine, peaks, "o", ms=5, color="#2a788e", mec="white", mew=1.0)
for r, col in zip(rhos_show, cols):
    i = int(np.argmin(np.abs(rhos_fine - r)))
    ax2.plot(rhos_fine[i], peaks[i], "o", ms=8, color=col, mec="white", mew=1.2, zorder=3)
for r in rhos_show:
    i = int(np.argmin(np.abs(rhos_fine - r)))
    ax2.annotate(f"{peaks[i]:.3f}", (rhos_fine[i], peaks[i]), textcoords="offset points",
                 xytext=(-4, 8), ha="right", fontsize=8, color="#333333")
ax2.set_xlabel(r"order $\varrho$"); ax2.set_ylabel(r"peak $|\xi(t,1/2)|$")
ax2.set_title("(b) peak displacement versus order", fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
Z = np.array([curves[float(r)] for r in rhos_fine])
lim = np.abs(Z).max()
im = ax3.pcolormesh(t, rhos_fine, Z, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), shading="auto")
cb = fig.colorbar(im, ax=ax3, fraction=0.05, pad=0.03); cb.set_label(r"$\xi(t,1/2)$", fontsize=9)
for r, col, ls in zip(rhos_show, cols, styles):
    ax3.axhline(r, color=col, lw=1.4, ls=ls)
ax3.set_xlabel(r"$t$"); ax3.set_ylabel(r"order $\varrho$"); ax3.grid(False)
ax3.set_title(r"(c) response map over $(t,\varrho)$", fontsize=9)

fig.savefig("figures/fig4_order.png", bbox_inches="tight", dpi=200); plt.show()
print({str(r): round(float(p), 4) for r, p in zip(rhos_fine, peaks)})

## Figure 5. fPINN under the recorded El Centro ground acceleration

In [ ]:
import matplotlib.ticker
ts = DATA["c_ts"]; w = DATA["s_w"]; xr, xp = DATA["e_xr"], DATA["e_xp"]; err = np.abs(xr - xp)
iw = int(np.argmin(np.abs(w - 0.5)))
print("fPINN relative L2 error (El Centro):", round(100 * np.linalg.norm(xr - xp) / np.linalg.norm(xr), 2), "%")
fig = plt.figure(figsize=(7.2, 4.9))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.5, wspace=0.45)
ax = fig.add_subplot(gs[0, :])
ax.plot(ts, xr[:, iw], color=C["blue"], lw=1.6, label="ABM (classical)")
ax.plot(ts[::3], xp[::3, iw], "o", ms=3.2, mfc="none", color=C["red"], label="fPINN")
ax.set_xlim(0, ts[-1]); ax.set_xlabel("physical time [s]"); ax.set_ylabel(r"$\xi(t,1/2)$")
ax.legend(ncol=2, fontsize=8, loc="lower left"); ax.set_title("(a) midpoint displacement under the El Centro record (no control)", fontsize=9)
ax2 = fig.add_subplot(gs[1, 0])
WW, TT = np.meshgrid(w, ts)
cs = ax2.contourf(TT, WW, 1e3 * err, 30, cmap="magma"); cb = fig.colorbar(cs, ax=ax2, fraction=0.05)
cb.locator = matplotlib.ticker.MaxNLocator(5); cb.update_ticks()
ax2.set_xlabel("physical time [s]"); ax2.set_ylabel(r"$w$"); ax2.grid(False)
ax2.set_title(r"(b) $|\xi_{\mathrm{ABM}}-\xi_{\mathrm{fPINN}}|\ (\times 10^{-3})$", fontsize=9)
ax3 = fig.add_subplot(gs[1, 1]); hl = DATA["e_hist"]
ax3.semilogy(hl[:, 0], hl[:, 1], color=C["green"], lw=1.4)
ax3.set_xlabel("iteration"); ax3.set_ylabel("loss"); ax3.set_title("(c) fPINN training loss", fontsize=9)
fig.savefig("figures/fig5_fpinn_elcentro.png", bbox_inches="tight", dpi=200); plt.show()

## Figure 6. Open-loop optimal control under the El Centro record (NN open-loop solver and direct method)

In [ ]:
ts = DATA["c_ts"]
fig, axs = plt.subplots(3, 1, figsize=(7.0, 5.8), sharex=True)
axs[0].plot(DATA["rec_t"], DATA["rec_a"], color="#bbbbbb", lw=0.7)
axs[0].plot(ts, DATA["c_ag"], color=C["grey"], lw=1.0)
axs[0].set_xlim(0, ts[-1]); axs[0].set_ylabel(r"$a_g$ [g]"); axs[0].set_title("(a) El Centro 1940, NS component (recorded)", fontsize=9)
axs[1].plot(ts, DATA["c_xb"], color=C["red"], lw=1.3, label="uncontrolled")
axs[1].plot(ts, DATA["c_xd"], color=C["amber"], lw=1.3, ls="--", label="direct method")
axs[1].plot(ts, DATA["c_xn"], color=C["blue"], lw=1.5, label="NN open-loop solver")
axs[1].set_ylabel(r"$\xi(t,1/2)$"); axs[1].legend(ncol=3, fontsize=8); axs[1].set_title("(b) midpoint displacement", fontsize=9)
axs[2].plot(ts, DATA["c_mud"], color=C["amber"], lw=1.2, ls="--", label="direct method")
axs[2].plot(ts, DATA["c_mun"], color=C["blue"], lw=1.5, label="NN open-loop solver")
axs[2].axvspan(0, float(DATA["kappa_s"]), color=C["grey"], alpha=0.2, lw=0)
axs[2].set_ylabel(r"$\mu(t)$"); axs[2].set_xlabel("physical time [s]"); axs[2].legend(fontsize=8)
axs[2].set_title(r"(c) optimal control (shaded: delay interval $[0,\kappa]$)", fontsize=9)
fig.tight_layout(); fig.savefig("figures/fig6_seismic_control.png", dpi=200); plt.show()

## Figure 7. Training cost and robustness estimate

In [ ]:
h_nn, h_d = DATA["h_nn"], DATA["h_d"]; J_unc = float(DATA["J_unc"]); Lam = float(DATA["Lam"])
gap, st = DATA["rob_gap"], DATA["rob_state"]
print("Final cost: NN =", round(np.nanmin(h_nn[~np.isnan(h_nn)][-1:]), 5), "| direct =", round(h_d[~np.isnan(h_d)][-1], 5), "| uncontrolled =", round(J_unc, 5))
print("Largest computed ratio =", round(float(np.max(st / gap)), 4), "< bound Lambda =", round(Lam, 2))
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8))
axs[0].semilogy(h_nn[~np.isnan(h_nn)], color=C["blue"], lw=1.4, label="NN open-loop solver")
axs[0].semilogy(h_d[~np.isnan(h_d)], color=C["amber"], lw=1.4, ls="--", label="direct method")
axs[0].axhline(J_unc, color=C["red"], lw=1.0, ls=":", label="uncontrolled")
axs[0].set_xlabel("iteration"); axs[0].set_ylabel(r"$\mathcal{J}(\mu)$"); axs[0].legend(fontsize=8); axs[0].set_title("(a) cost during training", fontsize=9)
axs[1].loglog(gap, st, "o-", color=C["blue"], ms=4, lw=1.3, label=r"computed $\|\xi^{\varpi^1}-\xi^{\varpi^2}\|_\ast$")
xx = np.array([gap.min(), gap.max()])
axs[1].loglog(xx, Lam * xx, color=C["red"], lw=1.3, ls="--", label=r"bound $\Lambda_\varpi\|\varpi^1-\varpi^2\|$")
axs[1].set_xlabel(r"$\|\varpi^1-\varpi^2\|_{L^2}$"); axs[1].legend(fontsize=8); axs[1].set_title("(b) robustness estimate", fontsize=9)
fig.tight_layout(); fig.savefig("figures/fig7_training_robustness.png", dpi=200); plt.show()

## Download all figures as one zip file

In [ ]:
import shutil
zip_path = shutil.make_archive("paper_figures", "zip", "figures")
print("Saved:", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    from IPython.display import FileLink, display; display(FileLink(zip_path))